# ATMASKOTS

In [ ]:
import re
import json
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# =========================
# CONFIG
# =========================
BASE = "https://www.delfi.lv"
SECTION = "https://www.delfi.lv/52061569/atmaskots"
PREFIX = "https://www.delfi.lv/52061569/atmaskots/"

TOTAL_PAGES = 22

SOURCE_NAME = "ATMASKOTS"
TOPIC_DEFAULT = "Atmaskots"
CLASS_LABEL_DEFAULT = 1

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "lv-LV,lv;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.delfi.lv/"
}

SLEEP_BETWEEN_PAGES = 0.35
SLEEP_BETWEEN_ARTICLES = 0.25

OUT_CSV = "delfi_atmaskots_raw.csv"
OUT_URLS_CSV = "delfi_atmaskots_urls.csv"

# =========================
# HELPERS
# =========================
def safe_get(url, timeout=25):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r

def normalize_whitespace(text: str) -> str:
    return " ".join(text.split()) if isinstance(text, str) else ""

def normalize_delfi_url(u: str) -> str:
    if not isinstance(u, str):
        return ""
    u = u.split("#")[0].strip()
    if u.endswith("/comments"):
        u = u[:-len("/comments")]
    return u

def get_atmaskots_urls_from_page(page_url: str):
    html = safe_get(page_url).text
    soup = BeautifulSoup(html, "html.parser")

    urls = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full = urljoin(BASE, href) if href.startswith("/") else href
        full = normalize_delfi_url(full)

        if full.startswith(PREFIX) and "/comments" not in full:
            if full != PREFIX.rstrip("/"):
                urls.add(full)

    return sorted(urls)

def discover_all_urls():
    all_unique = set()
    per_page = {}

    for page in range(1, TOTAL_PAGES + 1):
        page_url = SECTION if page == 1 else f"{SECTION}?page={page}"
        found = get_atmaskots_urls_from_page(page_url)

        before = len(all_unique)
        all_unique.update(found)
        added = len(all_unique) - before

        per_page[page] = {
            "page_url": page_url,
            "found_on_page": len(found),
            "new_unique_added": added,
            "total_unique_so_far": len(all_unique),
        }

        print(
            f"Page {page:02d}/{TOTAL_PAGES}: "
            f"found {len(found)}; new unique added {added}; total unique {len(all_unique)}"
        )
        time.sleep(SLEEP_BETWEEN_PAGES)

    return sorted(all_unique), per_page

def extract_from_jsonld(soup: BeautifulSoup) -> str:
    scripts = soup.find_all("script", attrs={"type": "application/ld+json"})
    for s in scripts:
        if not s.string:
            continue
        raw = s.string.strip()
        if not raw:
            continue
        try:
            data = json.loads(raw)
        except:
            continue

        objs = []
        if isinstance(data, dict):
            objs.append(data)
            if isinstance(data.get("@graph"), list):
                objs.extend([x for x in data["@graph"] if isinstance(x, dict)])
        elif isinstance(data, list):
            objs.extend([x for x in data if isinstance(x, dict)])

        for obj in objs:
            body = obj.get("articleBody")
            if isinstance(body, str):
                txt = normalize_whitespace(body)
                if len(txt) >= 200:
                    return txt
    return ""

def extract_publication_date(soup: BeautifulSoup) -> str:
    candidates = []

    for prop in ["article:published_time", "og:published_time"]:
        tag = soup.find("meta", attrs={"property": prop})
        if tag and tag.get("content"):
            candidates.append(tag["content"])

    tag = soup.find("meta", attrs={"itemprop": "datePublished"})
    if tag and tag.get("content"):
        candidates.append(tag["content"])

    t = soup.find("time")
    if t:
        if t.get("datetime"):
            candidates.append(t.get("datetime"))
        else:
            candidates.append(t.get_text(strip=True))

    for c in candidates:
        if not c:
            continue
        m = re.search(r"\d{4}-\d{2}-\d{2}", c)
        if m:
            return m.group(0)

    return ""

def extract_title(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        title = h1.get_text(" ", strip=True)
    else:
        og = soup.find("meta", attrs={"property": "og:title"})
        title = og["content"].strip() if og and og.get("content") else ""

    # remove trailing "(number)" (often comment count)
    title = re.sub(r"\(\s*\d+\s*\)\s*$", "", title).strip()
    return title

def extract_topic_if_available(soup: BeautifulSoup) -> str:
    meta = soup.find("meta", attrs={"property": "article:section"})
    if meta and meta.get("content"):
        t = meta["content"].strip()
        return t if t else TOPIC_DEFAULT
    return TOPIC_DEFAULT

def extract_main_text(soup: BeautifulSoup) -> str:
    txt = extract_from_jsonld(soup)
    if txt:
        return txt

    def remove_junk(root):
        junk_selectors = [
            "footer", "nav", "aside",
            "[class*='comment']", "[id*='comment']",
            "[class*='share']", "[class*='social']",
            "[class*='related']", "[class*='recommend']",
            "[class*='promo']", "[class*='subscribe']",
            "[class*='banner']", "[class*='ad']", "[id*='ad']",
            "[class*='cookie']", "[class*='consent']",
        ]
        for sel in junk_selectors:
            for node in root.select(sel):
                node.decompose()

    candidates = []

    article = soup.find("article")
    if article:
        remove_junk(article)
        ps = article.find_all("p")
        candidates.append(" ".join(p.get_text(" ", strip=True) for p in ps if p.get_text(strip=True)))

    body = soup.select_one('[itemprop="articleBody"]')
    if body:
        remove_junk(body)
        ps = body.find_all("p")
        candidates.append(" ".join(p.get_text(" ", strip=True) for p in ps if p.get_text(strip=True)))

    candidates = [normalize_whitespace(c) for c in candidates]
    candidates = [c for c in candidates if len(c) >= 200]
    return max(candidates, key=len) if candidates else ""

def scrape_one_article(url: str):
    soup = BeautifulSoup(safe_get(url).content, "html.parser")
    title = extract_title(soup)
    full_text = extract_main_text(soup)
    pub_date = extract_publication_date(soup)
    topic = extract_topic_if_available(soup)
    return title, full_text, pub_date, topic

# =========================
# RUN: 22 pages -> 1 raw CSV
# =========================
print("Step A: Discovering URLs across all 22 pages...")
urls, per_page_stats = discover_all_urls()

print("\nTotal unique URLs discovered:", len(urls))
pd.DataFrame({"url": urls}).to_csv(OUT_URLS_CSV, index=False, encoding="utf-8")
print("Saved URL list:", OUT_URLS_CSV)

print("\nStep B: Scraping all discovered URLs...")
records = []
N = len(urls)

for i, url in enumerate(urls, start=1):
    print(f"Scraping {i}/{N}: {url}")
    try:
        title, full_text, pub_date, topic = scrape_one_article(url)
        records.append({
            "title": title,
            "full_text": full_text,
            "source": SOURCE_NAME,
            "publication_date": pub_date,
            "topic": topic,
            "class_label": CLASS_LABEL_DEFAULT,
            "url": url,
            "text_length": len(full_text) if isinstance(full_text, str) else 0
        })
    except Exception as e:
        print("Error:", e)
    time.sleep(SLEEP_BETWEEN_ARTICLES)

df = pd.DataFrame(records)
df.to_csv(OUT_CSV, index=False, encoding="utf-8")

# =========================
# REPORT (overall + optional per-page)
# =========================
empty_n = (df["full_text"].fillna("").str.len() == 0).sum()
short_n = (df["full_text"].fillna("").str.len() < 200).sum()

print("\nSaved:", OUT_CSV)
print("Rows:", len(df))
print("Empty full_text:", empty_n)
print("Short (<200 chars):", short_n)

if len(df) > 0:
    print(
        "Min/Median/Max text length:",
        int(df["text_length"].min()),
        int(df["text_length"].median()),
        int(df["text_length"].max()),
    )

print("\nPer-page URL discovery summary:")
for page in range(1, TOTAL_PAGES + 1):
    s = per_page_stats[page]
    print(
        f"Page {page:02d}: found {s['found_on_page']}; "
        f"added {s['new_unique_added']}; total_unique {s['total_unique_so_far']}"
    )

print("\nDone.")

Step A: Discovering URLs across all 22 pages...
Page 01/22: found 22; new unique added 22; total unique 22
Page 02/22: found 24; new unique added 24; total unique 46
Page 03/22: found 25; new unique added 25; total unique 71
Page 04/22: found 28; new unique added 28; total unique 99
Page 05/22: found 22; new unique added 22; total unique 121
Page 06/22: found 23; new unique added 23; total unique 144
Page 07/22: found 25; new unique added 25; total unique 169
Page 08/22: found 20; new unique added 20; total unique 189
Page 09/22: found 29; new unique added 29; total unique 218
Page 10/22: found 29; new unique added 29; total unique 247
Page 11/22: found 29; new unique added 29; total unique 276
Page 12/22: found 29; new unique added 29; total unique 305
Page 13/22: found 30; new unique added 30; total unique 335
Page 14/22: found 30; new unique added 30; total unique 365
Page 15/22: found 29; new unique added 29; total unique 394
Page 16/22: found 16; new unique added 16; total unique 

# 1. Raw data inspection

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "delfi_atmaskots_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) BASIC INSPECTION
# --------------------------------------------------
print("\nFIRST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
        print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
        print(f"{i}. {title}")

print("\nTEXT LENGTH SUMMARY:")
if "full_text" in df.columns:
    lengths = df["full_text"].fillna("").astype(str).str.len()
    print("Min:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Max:", int(lengths.max()))

# --------------------------------------------------
# 2) TOP REPEATED SENTENCES
# --------------------------------------------------
print("\nTOP 40 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(40):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 3) TOKENIZATION
# --------------------------------------------------
def tokenize(text: str):
    return re.findall(r"[A-Za-zĀ-Žā-ž0-9\-]+", text.lower())

all_tokens = []
all_bigrams = []
all_trigrams = []

for text in df["full_text"].fillna("").astype(str):
    tokens = tokenize(text)
    tokens = [t for t in tokens if len(t) >= 4]

    all_tokens.extend(tokens)
    all_bigrams.extend(zip(tokens, tokens[1:]))
    all_trigrams.extend(zip(tokens, tokens[1:], tokens[2:]))

# --------------------------------------------------
# 4) TOP WORDS
# --------------------------------------------------
print("\nTOP 50 WORDS:")
word_counts = Counter(all_tokens)

for word, count in word_counts.most_common(50):
    print(f"{count:>4}  {word}")

# --------------------------------------------------
# 5) TOP BIGRAMS
# --------------------------------------------------
print("\nTOP 40 BIGRAMS:")
bigram_counts = Counter(all_bigrams)

for bg, count in bigram_counts.most_common(40):
    print(f"{count:>4}  {' '.join(bg)}")

# --------------------------------------------------
# 6) TOP TRIGRAMS
# --------------------------------------------------
print("\nTOP 40 TRIGRAMS:")
trigram_counts = Counter(all_trigrams)

for tg, count in trigram_counts.most_common(40):
    print(f"{count:>4}  {' '.join(tg)}")

# --------------------------------------------------
# 7) SOURCE / BRANDING CONTEXTS
# --------------------------------------------------
print("\nTOP 30 CONTEXTS CONTAINING SOURCE WORDS:")

SOURCE_PATTERNS = {
    "delfi": r'\b\S*delfi\S*\b',
    "atmaskots": r'\batmaskot\w*\b',
    "recheck": r'\bre:?check\b',
    "faktomats": r'\bfaktom[āa]ts\b',
    "tvnet": r'\b\S*tvnet\S*\b',
    "lsm": r'\bLSM(?:\.lv)?\b',
    "ltv": r'\bLTV(?:\.lv)?\b',
}

for label, pattern in SOURCE_PATTERNS.items():
    contexts = []

    for text in df["full_text"].fillna("").astype(str):
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(match.start() - 50, 0)
            end = min(match.end() + 100, len(text))
            context = text[start:end].strip()
            contexts.append(context)

    if contexts:
        print(f"\n--- {label.upper()} ---")
        context_counts = Counter(contexts)
        for context, count in context_counts.most_common(15):
            print(f"{count:>4}  {context}")

# --------------------------------------------------
# 8) FACT-CHECK CUE FAMILY COUNTS
# Uses stems so Latvian inflections are included
# --------------------------------------------------
print("\nFACT-CHECK CUE FAMILY COUNTS (TEXT):")

CUE_FAMILIES = {
    "faktu pārbaud*": r"\bfaktu\s+pārbaud\w*\b",
    "atmaskot*": r"\batmaskot\w*\b",
    "maldin*": r"\bmaldin\w*\b",
    "nepaties*": r"\bnepaties\w*\b",
    "paties*": r"\bpaties\w*\b",
    "apgalvoj*": r"\bapgalvoj\w*\b",
    "secin*": r"\bsecin\w*\b",
    "kontekst*": r"\bkontekst\w*\b",
    "dezinform*": r"\bdezinform\w*\b",
    "propagand*": r"\bpropagand\w*\b",
    "fakts/faktu": r"\bfakt\w*\b",
}

full_text_series = df["full_text"].fillna("").astype(str)

for label, pattern in CUE_FAMILIES.items():
    count = full_text_series.str.contains(pattern, case=False, regex=True).sum()
    print(f"{count:>4}  {label}")

print("\nFACT-CHECK CUE FAMILY COUNTS (TITLES):")

title_series = df["title"].fillna("").astype(str)

for label, pattern in CUE_FAMILIES.items():
    count = title_series.str.contains(pattern, case=False, regex=True).sum()
    print(f"{count:>4}  {label}")

# --------------------------------------------------
# 9) TOP TITLE PREFIXES BEFORE COLON
# --------------------------------------------------
print("\nTOP TITLE PREFIXES BEFORE COLON:")

prefixes = []

for title in title_series:
    match = re.match(r"^\s*([^:]{1,50}):", title)
    if match:
        prefixes.append(match.group(1).strip())

prefix_counts = Counter(prefixes)

for prefix, count in prefix_counts.most_common(30):
    print(f"{count:>4}  {prefix}:")

# --------------------------------------------------
# 10) EXACT KEYWORD CHECKS
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS:")

keywords = [
    "delfi",
    "atmaskots",
    "faktu pārbaude",
    "nav taisnība",
    "trūkst konteksta",
    "secinājums",
    "apgalvojums",
    "maldina",
    "nepatiesa",
]

for k in keywords:
    text_count = full_text_series.str.contains(k, case=False, regex=False).sum()
    title_count = title_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>20} | text: {int(text_count):>4} | title: {int(title_count):>4}")

ROWS: 472
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'text_length']

FIRST 5 TITLES:
1. 'Tiek pūsta migla acīs visai sabiedrībai’: kā Kremlis dezinformē par Navaļniju un Putinu
2. Faktu pārbaude: Vai ES tiešām plāno aizliegt vecu automašīnu remontu un īstenot ļaužu imobilizāciju
3. Faktu pārbaude: Baltimoras tilta sabrukšana nonāk sazvērestības teoriju krustugunīs
4. Faktu pārbaude: Latvijas teritoriju senvēsturē nekad nav apdzīvojuši slāvi
5. 'Liels spainis, no kā pasmelties teorijas': kāpēc Baltimoras tilta sabrukšanu maldīgi saistīja pat ar Covid-19

LAST 5 TITLES:
1. Faktu pārbaude: Latvija neplāno deportēt tos Krievijas pilsoņus, kas balsos Krievijas prezidenta vēlēšanās
2. Faktu pārbaude: Bērnudārzos nav paredzēts mācīt bērniem uzvilkt prezervatīvu uz burkāna
3. Faktu pārbaude: vai viedie elektrības skaitītāji mūs izseko?
4. Faktu pārbaude: Vai NATO pastāvēšanas pamats zudis reizē ar Varšavas paktu?
5. Uzņēmēja Maska šaubas par NA

Based on the Step 1 leakage inspection, repeated funding notices, navigation phrases, outlet branding, and section labels were assigned to source cleaning, because they constituted non-editorial boilerplate or source-identifying text. In contrast, verdict-related and fact-checking cue expressions, such as “nav taisnība”, “nepaties*”, “maldin*”, “apgalvoj*”, and “secin*”, were assigned to the masking stage, because they directly signalled the fact-checking function and could leak class-specific information to the model.

# 2. Cleaning

In [ ]:
%%writefile clean_delfi_atmaskots.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "delfi_atmaskots_raw.csv"
OUT_FILE = "delfi_atmaskots_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup
# --------------------------------------------------
TITLE_PREFIX_PATTERNS = [
    r'^\s*Faktu\s+pārbaude\s*:\s*',
    r'^\s*Atmaskots\s*:\s*',
    r'^\s*Video\s*:\s*',
]
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")

# --------------------------------------------------
# Boilerplate / promo / funding / tail fragments
# --------------------------------------------------
REMOVE_PATTERNS = [
    r'\*?\s*Faktu\s+pārbaudes\s+materiālu\s+finansē\s+Eiropas\s+Mediju\s+un\s+informācijas\s+fonds\s+\(EMIF\),?\s+ko\s+pārvalda\s+"?Calouste\s+Gulbenkian\s+Foundation"?\.?',
    r'Visus\s+rakstus\s+par\s+dezinformāciju\s+var\s+lasīt\s+sadaļā\s+"?\s*Atmaskots\s*"?\.?',
    r'Visus\s+"?\s*Delfi\s*"?\s+faktu\s+pārbaudes\s+rakstus\s+var\s+lasīt\s+sadaļā\s+"?\s*Atmaskots\s*"?\.?',
    r'Savukārt\s+visus\s+"?\s*Delfi\s*"?\s+faktu\s+pārbaudes\s+rakstus\s+var\s+lasīt\s+sadaļā\s+"?\s*Atmaskots\s*"?\.?',
    r'Visus\s+"?\s*Delfi\s*"?\s+rakstus\s+par\s+dezinformāciju\s+var\s+lasīt\s+sadaļā\s+"?\s*Atmaskots\s*"?\.?',
    r'"?\s*Delfi\s*"?\s+ir\s+EDMO\s+ietvaros\s+radītā\s+Baltijas\s+Iesaistes\s+centra\s+cīņai\s+pret\s+informācijas\s+traucējumiem\s+\(BECID\)\s+līdzdibinātājs\.?',
    r'Faktu\s+pārbaudi\s+līdzfinansē\s+Eiropas\s+Digitālo\s+mediju\s+observatorija\s+\(EDMO\)\.?',
    r'Satura\s+projektu\s+līdzfinansē\s+Vācijas\s+Federatīvā\s+Republika\s+konkursa\s+par\s+noturīgumu\s+un\s+medijpratības\s+veicināšanu\s+2024\.?',
    r'EMIF\s+(?:faktu\s+pārbaudes\s+)?principi\s+ir\s+saistoši\s+arī\s+izdevumam\s+[“"]?The\s+Journal[”"]?\.?',
    r'Par\s+saturu\s+atbild\s+AS\s+"?\s*DELFI\s*"?\.?',
    r'Klaus(?:ies|ieties)\s+arī\s+"?Atmaskots(?:\.lv)?"?\s+podkāst(?:u|us)?(?:\s+šeit)?\.?',
    r'Pilnu\s+raidījuma\s+"?Komandcentrs"?\s+epizodi[^\.!\?]*',
    r'Skat(?:\s+\d{4})?\s+pieejams\s+https?://\S+',
    r'pieejams\s+https?://\S+',
    r'https?://archive\.\S+',
    r'https?://rubaltic\.\S+',
    r'jūlijs,\s*Sanktpēterburga,\s*http://kremlin\.ru/events/president/news/\d+\.?',
    r'Заседание\s+совета\s+глав\s+государств\s+[–-]\s+членов\s+ШОС\.?',
]

# Cut long tails from first marker to end
TAIL_MARKERS = [
    r'Faktu\s+pārbaudes\s+materiālu\s+finansē\s+Eiropas\s+Mediju\s+un\s+informācijas\s+fonds',
    r'Visus\s+rakstus\s+par\s+dezinformāciju\s+var\s+lasīt\s+sadaļā',
    r'Visus\s+"?\s*Delfi\s*"?\s+faktu\s+pārbaudes\s+rakstus\s+var\s+lasīt\s+sadaļā',
    r'Savukārt\s+visus\s+"?\s*Delfi\s*"?\s+faktu\s+pārbaudes\s+rakstus\s+var\s+lasīt\s+sadaļā',
    r'Visus\s+"?\s*Delfi\s*"?\s+rakstus\s+par\s+dezinformāciju',
    r'Klaus(?:ies|ieties)\s+arī\s+"?Atmaskots(?:\.lv)?"?\s+podkāst',
    r'Pilnu\s+raidījuma\s+"?Komandcentrs"?\s+epizodi',
    r'Par\s+saturu\s+atbild\s+AS\s+"?\s*DELFI\s*"?',
]

# --------------------------------------------------
# Source / branding removal
# --------------------------------------------------
SOURCE_PATTERNS = [
    r'[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'[\"“”„\']\s*Atmaskots\s*[\"“”„\']',
    r'[\"“”„\']\s*Atmaskots\.lv\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\.ee\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\.lv\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\s+Bizness\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\s+Plus\s*[\"“”„\']',
    r'[\"“”„\']\s*Spried\s+ar\s+Delfi\s*[\"“”„\']',
    r'portāls\s+[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'portālam\s+[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'portālā\s+[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'portālu\s+[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'igauņu\s+portāls\s+[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'\bDelfi\.lv\b',
    r'\bDelfi\.ee\b',
    r'\bDelfi\.lt\b',
    r'\bAtmaskots\.lv\b',
    r'\bSpried\s+ar\s+Delfi\b',
    r'\bDelfi\s+Bizness\b',
]

COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    text = re.sub(r"<[^>]+>", " ", text)

    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def cut_from_first_marker(text: str, markers: list[str]) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    positions = []
    for pat in markers:
        m = re.search(pat, text, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
        if m:
            positions.append(m.start())

    if not positions:
        return text

    return text[:min(positions)].strip()


def clean_title(title: object) -> str:
    title = normalize_text(title)

    for pat in TITLE_PREFIX_PATTERNS:
        title = re.sub(pat, "", title, flags=re.IGNORECASE | re.UNICODE)

    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title


def clean_full_text(text: object) -> str:
    text = normalize_text(text)

    # First cut long tails
    text = cut_from_first_marker(text, TAIL_MARKERS)

    # Then remove inline boilerplate and branding
    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Cleanup leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Overwriting clean_delfi_atmaskots.py


In [ ]:
!python clean_delfi_atmaskots.py

Saved: delfi_atmaskots_clean.csv
Rows before: 472
Rows after: 430
Removed rows: 42

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'text_length']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


# 3. Masking

In [ ]:
%%writefile mask_delfi_atmaskots.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "delfi_atmaskots_clean.csv"
OUT_FILE = "delfi_atmaskots_masked.csv"
AUDIT_FILE = "delfi_atmaskots_mask_audit.csv"

MASK_TOKEN = "[MASK]"

# --------------------------------------------------
# Title masking:
# mask label-signaling prefixes and verdict cues
# --------------------------------------------------
TITLE_MASK_PATTERNS = [
    # common fact-check prefixes
    r'^\s*Faktu\s+pārbaude\s*:\s*',
    r'^\s*Atmaskots\s*:\s*',
    r'^\s*Video\s*:\s*',

    # verdict-like title starts
    r'^\s*Nav\s+taisn(?:ība|i)\s*[:\-–—]?\s*',
    r'^\s*Trūkst\s+kontekst\w*\s*[:\-–—]?\s*',
    r'^\s*Secin\w*\s*[:\-–—]?\s*',
    r'^\s*Apgalvoj\w*\s*[:\-–—]?\s*',
    r'^\s*Nepaties\w*\s*[:\-–—]?\s*',
    r'^\s*Maldin\w*\s*[:\-–—]?\s*',
]

# --------------------------------------------------
# Text masking:
# mask verdict / fact-check cue families
# --------------------------------------------------
TEXT_MASK_PATTERNS = [
    # longest / most explicit first
    r'\bnav\s+taisn(?:ība|i)\b',
    r'\btrūkst\s+kontekst\w*\b',
    r'\bfaktu\s+pārbaud\w*\b',
    r'\bnepaties\w*\b',
    r'\bmaldin\w*\b',
    r'\bapgalvoj\w*\b',
    r'\bsecin\w*\b',
    r'\bkontekst\w*\b',
    r'\bpaties\w*\b',
    r'\bdezinform\w*\b',
    r'\bpropagand\w*\b',
]

COMPILED_TITLE_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TITLE_MASK_PATTERNS
]

COMPILED_TEXT_MASK_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in TEXT_MASK_PATTERNS
]


def safe_text(text: object) -> str:
    if pd.isna(text):
        return ""
    return str(text).strip()


def cleanup_masked_text(text: str) -> str:
    # collapse repeated masks
    text = re.sub(r'(\[MASK\]\s*){2,}', MASK_TOKEN + " ", text)

    # merge awkward mask chains
    text = re.sub(r'\[MASK\]\s*[,;:]+\s*\[MASK\]', MASK_TOKEN, text)
    text = re.sub(r'\[MASK\]\s*[-–—:]\s*\[MASK\]', MASK_TOKEN, text)

    # remove awkward leading punctuation after mask replacement
    text = re.sub(r'^\s*[,;:.\-–—]+\s*', "", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def mask_title(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TITLE_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN + " ", text)

    # also mask verdict cue words if they remain inside title
    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)

    # normalize beginnings like "[MASK] - ..."
    text = re.sub(r"^\[MASK\]\s*[:\-–—]\s*", MASK_TOKEN + " ", text)

    return text.strip()


def mask_full_text(text: object) -> str:
    text = safe_text(text)

    for pattern in COMPILED_TEXT_MASK_PATTERNS:
        text = pattern.sub(MASK_TOKEN, text)

    text = cleanup_masked_text(text)

    return text.strip()


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Apply masking
# ----------------------------------------
df["title_masked"] = df["title_clean"].apply(mask_title)
df["full_text_masked"] = df["full_text_clean"].apply(mask_full_text)

# ----------------------------------------
# Mask diagnostics
# ----------------------------------------
df["title_changed_by_masking"] = df["title_clean"] != df["title_masked"]
df["text_changed_by_masking"] = df["full_text_clean"] != df["full_text_masked"]

df["title_mask_count"] = df["title_masked"].fillna("").astype(str).str.count(r"\[MASK\]")
df["text_mask_count"] = df["full_text_masked"].fillna("").astype(str).str.count(r"\[MASK\]")

# ----------------------------------------
# Save main masked file
# ----------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Save audit file: only changed rows
# ----------------------------------------
audit_df = df[
    df["title_changed_by_masking"] | df["text_changed_by_masking"]
][[
    "title",
    "title_clean",
    "title_masked",
    "full_text",
    "full_text_clean",
    "full_text_masked",
    "title_mask_count",
    "text_mask_count",
]].copy()

audit_df.to_csv(AUDIT_FILE, index=False, encoding="utf-8")

# ----------------------------------------
# Report
# ----------------------------------------
print("Saved:", OUT_FILE)
print("Saved audit:", AUDIT_FILE)
print("Rows:", rows_before)

print("\nMask summary:")
print({
    "title_changed_rows": int(df["title_changed_by_masking"].sum()),
    "text_changed_rows": int(df["text_changed_by_masking"].sum()),
    "total_title_masks": int(df["title_mask_count"].sum()),
    "total_text_masks": int(df["text_mask_count"].sum()),
})

print("\nNew columns added:")
print([
    "title_masked",
    "full_text_masked",
    "title_changed_by_masking",
    "text_changed_by_masking",
    "title_mask_count",
    "text_mask_count",
])

Overwriting mask_delfi_atmaskots.py


In [ ]:
!python dedup_delfi_atmaskots.py

Saved: delfi_atmaskots_dedup.csv
Rows before dedup: 431
Rows after URL dedup: 431
Rows after text dedup: 431
Total removed: 0


In [ ]:
!python mask_delfi_atmaskots.py

Saved: delfi_atmaskots_masked.csv
Saved audit: delfi_atmaskots_mask_audit.csv
Rows: 430

Mask summary:
{'title_changed_rows': 92, 'text_changed_rows': 370, 'total_title_masks': 97, 'total_text_masks': 2385}

New columns added:
['title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count']


# 4. Remove mask

In [ ]:
%%writefile remove_mask_delfi_atmaskots.py
from __future__ import annotations

import re
import pandas as pd

IN_FILE = "delfi_atmaskots_masked.csv"
OUT_FILE = "delfi_atmaskots_nomask.csv"

MASK_PATTERN = r"\[MASK\]"


def remove_mask_tokens(text: object) -> str:
    if pd.isna(text):
        return ""

    text = str(text)

    # remove mask token
    text = re.sub(MASK_PATTERN, " ", text, flags=re.IGNORECASE)

    # remove awkward leftover punctuation at start
    text = re.sub(r"^\s*[,;:.\-–—]+\s*", "", text)

    # remove awkward punctuation fragments after removal
    text = re.sub(r"\s*[-–—:]\s*(?=[,.;!?)]|$)", " ", text)
    text = re.sub(r"([.!?])\s*[,;:.\-–—]+\s*", r"\1 ", text)

    # remove empty brackets
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

before_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
before_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df["title_masked"] = df["title_masked"].apply(remove_mask_tokens)
df["full_text_masked"] = df["full_text_masked"].apply(remove_mask_tokens)

after_title_masks = df["title_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()
after_text_masks = df["full_text_masked"].fillna("").astype(str).str.count(MASK_PATTERN).sum()

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows:", rows_before)
print("\nMask removal summary:")
print({
    "title_masks_before": int(before_title_masks),
    "text_masks_before": int(before_text_masks),
    "title_masks_after": int(after_title_masks),
    "text_masks_after": int(after_text_masks),
})

Overwriting remove_mask_delfi_atmaskots.py


In [ ]:
!python remove_mask_delfi_atmaskots.py

Saved: delfi_atmaskots_nomask.csv
Rows: 430

Mask removal summary:
{'title_masks_before': 97, 'text_masks_before': 2385, 'title_masks_after': 0, 'text_masks_after': 0}


5. Source level dedublication

In [ ]:
%%writefile dedup_delfi_atmaskots.py
from __future__ import annotations

import pandas as pd

IN_FILE = "delfi_atmaskots_nomask.csv"
OUT_FILE = "delfi_atmaskots_dedup.csv"


df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Duplicate detection
# ----------------------------------------

df["is_duplicate_url"] = df.duplicated(subset=["url"], keep="first")

df["is_duplicate_text"] = df.duplicated(
    subset=["title_clean", "full_text_clean"],
    keep="first"
)

# ----------------------------------------
# Remove duplicates
# ----------------------------------------

df = df.drop_duplicates(subset=["url"], keep="first")

df = df.drop_duplicates(
    subset=["title_clean", "full_text_clean"],
    keep="first"
)

rows_after = len(df)

df = df.reset_index(drop=True)

# ----------------------------------------
# Save
# ----------------------------------------

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)

print("\nRows before:", rows_before)
print("Rows after:", rows_after)
print("Removed duplicates:", rows_before - rows_after)

print("\nDuplicate flags:")
print({
    "duplicate_url": int(df["is_duplicate_url"].sum()),
    "duplicate_text": int(df["is_duplicate_text"].sum())
})

Overwriting dedup_delfi_atmaskots.py


In [ ]:
!python dedup_delfi_atmaskots.py

Saved: delfi_atmaskots_dedup.csv

Rows before: 430
Rows after: 430
Removed duplicates: 0

Duplicate flags:
{'duplicate_url': 0, 'duplicate_text': 0}


6. Source level audit

In [ ]:
%%writefile audit_delfi_atmaskots.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "delfi_atmaskots_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Text length summary
# --------------------------------------------------
print("\nTEXT LENGTH SUMMARY:")
lengths = df["full_text_clean"].fillna("").astype(str).str.len()
print("Min:", int(lengths.min()))
print("Median:", float(lengths.median()))
print("Max:", int(lengths.max()))

# --------------------------------------------------
# 2) Top repeated sentences
# --------------------------------------------------
print("\nTOP 40 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_masked"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(40):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 3) Leakage / source words still left
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TEXT):")

TEXT_KEYWORDS = [
    "delfi",
    "atmaskots",
    "faktu pārbaude",
    "nav taisnība",
    "trūkst konteksta",
    "secinājums",
    "apgalvojums",
    "maldina",
    "nepatiesa",
    "emif",
    "edmo",
]

text_series = df["full_text_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = text_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>20} : {int(count)}")

# --------------------------------------------------
# 4) Leakage / source words in titles
# --------------------------------------------------
print("\nEXACT KEYWORD CHECKS (TITLES):")

title_series = df["title_masked"].fillna("").astype(str)

for k in TEXT_KEYWORDS:
    count = title_series.str.contains(k, case=False, regex=False).sum()
    print(f"{k:>20} : {int(count)}")

# --------------------------------------------------
# 5) Contexts for remaining DELFI / ATMASKOTS / FACT-CHECK words
# --------------------------------------------------
print("\nTOP CONTEXTS FOR REMAINING SOURCE WORDS:")

SOURCE_PATTERNS = {
    "delfi": r'\b\S*delfi\S*\b',
    "atmaskots": r'\batmaskot\w*\b',
    "faktu pārbaude": r'\bfaktu\s+pārbaude\b',
    "emif": r'\bemif\b',
    "edmo": r'\bedmo\b',
}

for label, pattern in SOURCE_PATTERNS.items():
    contexts = []

    for text in text_series:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(match.start() - 50, 0)
            end = min(match.end() + 100, len(text))
            context = text[start:end].strip()
            contexts.append(context)

    if contexts:
        print(f"\n--- {label.upper()} ---")
        context_counts = Counter(contexts)
        for context, count in context_counts.most_common(20):
            print(f"{count:>4}  {context}")

# --------------------------------------------------
# 6) Masking diagnostics
# --------------------------------------------------
if "title_mask_count" in df.columns and "text_mask_count" in df.columns:
    print("\nMASKING SUMMARY:")
    print("Rows with title masks:", int((df["title_mask_count"] > 0).sum()))
    print("Rows with text masks:", int((df["text_mask_count"] > 0).sum()))
    print("Total title masks:", int(df["title_mask_count"].sum()))
    print("Total text masks:", int(df["text_mask_count"].sum()))

# --------------------------------------------------
# 7) Empty checks
# --------------------------------------------------
print("\nEMPTY TEXTS:")
print(int((df["full_text_masked"].fillna("").astype(str).str.strip() == "").sum()))

print("\nEMPTY TITLES:")
print(int((df["title_masked"].fillna("").astype(str).str.strip() == "").sum()))

Overwriting audit_delfi_atmaskots.py


In [ ]:
!python audit_delfi_atmaskots.py

ROWS: 430
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean', 'title_masked', 'full_text_masked', 'title_changed_by_masking', 'text_changed_by_masking', 'title_mask_count', 'text_mask_count', 'is_duplicate_url', 'is_duplicate_text']

TEXT LENGTH SUMMARY:
Min: 315
Median: 3206.0
Max: 24334

TOP 40 REPEATED SENTENCES:
   4  Kā vienu no iebrukuma mērķiem Krievija sākotnēji minēja Ukrainas "denacifikāciju".
   4  Krievijas attieksmi pret ukraiņiem, tostarp nogalinot un veidojot filtrācijas nometnes, raksturo šis "fašisma manifests", ko publicēja Kremļa ziņu aģentūra "RIA Novosti".
   4  gada, kad Krievija okupēja Krimas pussalu un aizsāka karu Ukrainas austrumos.
   3  Krievijas pilna mēroga iebrukums Ukrainā ar raķešu apšaudēm, uzlidojumiem un sauszemes spēku ienākšanu sākās agrā 24.
   3  Okupanti Ukrainā cietuši smagus zaudējumus un tiek apsūdzēti apzinātā civiliedzīvotāju noga